In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

/Users/aditya/Desktop/Pandas/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# Browser-like headers help the request look like it is coming from a normal browser session.
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.ambitionbox.com/",
    "Connection": "keep-alive",
}

In [10]:
# Fetch the page HTML using the browser-style headers.
url = "https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav"
response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()

# Parse the company cards from the server-rendered HTML.
soup = BeautifulSoup(response.text, "html.parser")
company_cards = []

for card in soup.select("div.companyCardWrapper"):
    text = card.get_text(" ", strip=True)
    match = pd.Series(text).str.extract(
        r"^(?P<company>.+?) Follow (?P<rating>\d\.\d) \((?P<reviews>[^)]+)\) (?P<industry>.+?) \| (?P<location>.+?) (?:Highly Rated For|Critically Rated For)",
        expand=True,
    ).iloc[0]

    location = match.get("location")
    if isinstance(location, str):
        location = location.split(" AmbitionBox")[0].strip()

    company_cards.append({
        "company": card.select_one("h2.companyCardWrapper__companyName").get_text(strip=True) if card.select_one("h2.companyCardWrapper__companyName") else None,
        "rating": card.select_one(".companyCardWrapper__companyRating").get_text(strip=True) if card.select_one(".companyCardWrapper__companyRating") else None,
        "reviews": card.select_one(".companyCardWrapper__companyRatingCount").get_text(" ", strip=True).strip("()").strip() if card.select_one(".companyCardWrapper__companyRatingCount") else None,
        "industry": match.get("industry"),
        "location": location,
        "company_url": card.select_one("meta[itemprop='url']").get("content") if card.select_one("meta[itemprop='url']") else None,
    })

companies = pd.DataFrame(company_cards)
companies.head(10)

,company,rating,reviews,industry,location,company_url
0,TCS,3.3,1.2L,IT Services & Consulting,Bengaluru +448 other locations,https://www.ambitionbox.com/overview/tcs-overview
1,Accenture,3.7,75k,IT Services & Consulting,Bengaluru +261 other locations,https://www.ambitionbox.com/overview/accenture...
2,Wipro,3.6,66.3k,IT Services & Consulting,Hyderabad +375 other locations,https://www.ambitionbox.com/overview/wipro-ove...
3,Cognizant,3.7,62.6k,IT Services & Consulting,Hyderabad +233 other locations,https://www.ambitionbox.com/overview/cognizant...
4,Capgemini,3.6,54.6k,IT Services & Consulting,Bengaluru +188 other locations,https://www.ambitionbox.com/overview/capgemini...
5,HDFC Bank,3.8,53.7k,Banking,Mumbai +1863 other locations,https://www.ambitionbox.com/overview/hdfc-bank...
6,Infosys,3.5,49.8k,IT Services & Consulting,Bengaluru +247 other locations,https://www.ambitionbox.com/overview/infosys-o...
7,HCLTech,3.4,46.9k,IT Services & Consulting,Chennai +232 other locations,https://www.ambitionbox.com/overview/hcl-techn...
8,ICICI Bank,3.9,46.7k,Banking,Mumbai +1447 other locations,https://www.ambitionbox.com/overview/icici-ban...
9,Tech Mahindra,3.3,44.2k,IT Services & Consulting,Hyderabad +323 other locations,https://www.ambitionbox.com/overview/tech-mahi...
